# Task 2: EDA on Retail Sales Data

**Track:** Data Analytics - Level 1
**Objective:** Perform a thorough Exploratory Data Analysis on a retail sales dataset to uncover patterns, customer behaviour trends, and actionable business insights.

**Tech Stack:** Python, pandas, matplotlib, seaborn, Jupyter Notebook

## 1. Load Dataset & Initial Inspection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Load dataset
df = pd.read_csv('retail_sales_data.csv')
df['order_date'] = pd.to_datetime(df['order_date'])

print(f'Dataset Shape: {df.shape}')
print(f'\nFirst 5 rows:')
display(df.head())
print(f'\nColumn Names: {list(df.columns)}')
print(f'\nData Types:')
print(df.dtypes)
print(f'\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

## 2. Data Quality Check

In [ ]:
# Null value check
print('=== NULL VALUES ===')
null_report = pd.DataFrame({
    'Null Count': df.isnull().sum(),
    'Null %': (df.isnull().sum() / len(df) * 100).round(2)
})
display(null_report[null_report['Null Count'] > 0])

# Duplicate check
dup_count = df.duplicated().sum()
print(f'\nDuplicate rows: {dup_count}')

# Basic statistics for numerical columns
print('\n=== DESCRIPTIVE STATISTICS ===')
display(df.describe())

## 3. Handle Missing Values

In [ ]:
# Handle missing values
df_clean = df.copy()

# customer_age: impute with median
age_median = df_clean['customer_age'].median()
df_clean['customer_age'] = df_clean['customer_age'].fillna(age_median)
print(f'Customer age imputed with median: {age_median}')

# customer_gender: impute with mode
gender_mode = df_clean['customer_gender'].mode()[0]
df_clean['customer_gender'] = df_clean['customer_gender'].fillna(gender_mode)
print(f'Customer gender imputed with mode: {gender_mode}')

# category: impute with mode
cat_mode = df_clean['category'].mode()[0]
df_clean['category'] = df_clean['category'].fillna(cat_mode)
print(f'Category imputed with mode: {cat_mode}')

print(f'\nRemaining nulls:\n{df_clean.isnull().sum()}')
print(f'Cleaned shape: {df_clean.shape}')

## 4. Descriptive Statistics

In [ ]:
# Descriptive statistics for all numerical columns
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns

stats = pd.DataFrame()
for col in numeric_cols:
    stats[col] = [
        df_clean[col].mean(),
        df_clean[col].median(),
        df_clean[col].mode().iloc[0] if not df_clean[col].mode().empty else np.nan,
        df_clean[col].std()
    ]
stats.index = ['Mean', 'Median', 'Mode', 'Std Dev']
print('=== NUMERICAL COLUMNS STATISTICS ===')
display(stats.round(2))

## 5. Time Series Analysis - Monthly & Quarterly Sales Trends

In [ ]:
# Monthly sales trends
df_clean['year_month'] = df_clean['order_date'].dt.to_period('M')
monthly_sales = df_clean.groupby('year_month')['total_amount'].sum().reset_index()
monthly_sales['year_month_str'] = monthly_sales['year_month'].astype(str)

# Quarterly sales trends
df_clean['year_quarter'] = df_clean['order_date'].dt.to_period('Q')
quarterly_sales = df_clean.groupby('year_quarter')['total_amount'].sum().reset_index()
quarterly_sales['year_quarter_str'] = quarterly_sales['year_quarter'].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Monthly trend
axes[0].plot(monthly_sales['year_month_str'], monthly_sales['total_amount'], marker='o', linewidth=2, markersize=4)
axes[0].set_title('Monthly Sales Trend', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Total Sales ($)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3)

# Quarterly trend
axes[1].plot(quarterly_sales['year_quarter_str'], quarterly_sales['total_amount'], marker='s', linewidth=2, markersize=6, color='orange')
axes[1].set_title('Quarterly Sales Trend', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Quarter')
axes[1].set_ylabel('Total Sales ($)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('=== MONTHLY SALES SUMMARY ===')
display(monthly_sales)
print('\n=== QUARTERLY SALES SUMMARY ===')
display(quarterly_sales)

### Observations - Time Series

- **Monthly trends** show [insert observation from chart]
- **Quarterly trends** reveal [insert observation from chart]
- Seasonal patterns are [evident/not evident] in the data
- Peak sales months: [to be filled after viewing chart]

## 6. Customer Demographics Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Age distribution
axes[0, 0].hist(df_clean['customer_age'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Customer Age Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df_clean['customer_age'].mean(), color='red', linestyle='--', label=f'Mean: {df_clean["customer_age"].mean():.1f}')
axes[0, 0].legend()

# Gender breakdown
gender_counts = df_clean['customer_gender'].value_counts()
axes[0, 1].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%', startangle=90)
axes[0, 1].set_title('Customer Gender Distribution', fontweight='bold')

# Age by gender boxplot
sns.boxplot(data=df_clean, x='customer_gender', y='customer_age', ax=axes[1, 0])
axes[1, 0].set_title('Age Distribution by Gender', fontweight='bold')

# Average spend by age group
df_clean['age_group'] = pd.cut(df_clean['customer_age'], bins=[0, 25, 35, 50, 65, 100], labels=['18-25', '26-35', '36-50', '51-65', '65+'])
avg_spend_age = df_clean.groupby('age_group')['total_amount'].mean().reset_index()
axes[1, 1].bar(avg_spend_age['age_group'].astype(str), avg_spend_age['total_amount'])
axes[1, 1].set_title('Average Spend by Age Group', fontweight='bold')
axes[1, 1].set_xlabel('Age Group')
axes[1, 1].set_ylabel('Average Spend ($)')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('=== GENDER COUNTS ===')
print(gender_counts)
print(f'\n=== AVERAGE SPEND BY AGE GROUP ===')
display(avg_spend_age)

### Observations - Demographics

- Age distribution is [describe shape - normal, skewed, etc.]
- Gender split: [describe breakdown]
- Age vs spend relationship: [describe pattern]
- Key demographic segment: [identify highest spending group]

## 7. Product Analysis

In [ ]:
# Top 10 best-selling products by revenue
top_products = df_clean.groupby('product_name').agg({
    'total_amount': 'sum',
    'quantity': 'sum',
    'order_id': 'count'
}).rename(columns={'order_id': 'order_count'}).sort_values('total_amount', ascending=False).head(10)

# Revenue by category
category_revenue = df_clean.groupby('category').agg({
    'total_amount': 'sum',
    'quantity': 'sum',
    'order_id': 'count'
}).rename(columns={'order_id': 'order_count'}).sort_values('total_amount', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 10 products
top_products['total_amount'].sort_values(ascending=True).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 10 Best-Selling Products by Revenue', fontweight='bold')
axes[0].set_xlabel('Total Revenue ($)')

# Revenue by category
category_revenue['total_amount'].sort_values(ascending=True).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Revenue by Product Category', fontweight='bold')
axes[1].set_xlabel('Total Revenue ($)')

plt.tight_layout()
plt.show()

print('=== TOP 10 PRODUCTS ===')
display(top_products)
print('\n=== REVENUE BY CATEGORY ===')
display(category_revenue)

### Observations - Product Analysis

- Top product: [name] with revenue of $[amount]
- Top category: [name] contributing [X]% of total revenue
- [Number] categories account for [X]% of revenue (Pareto principle)

## 8. Correlation Heatmap

In [ ]:
# Correlation matrix for numerical variables
numeric_df = df_clean.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix - Numerical Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('=== CORRELATION MATRIX ===')
display(corr_matrix.round(2))

### Observations - Correlations

- Strongest positive correlation: [variables] (r = [value])
- Strongest negative correlation: [variables] (r = [value])
- Key insight: [describe meaningful correlation for business]

## 9. Additional Visualization - Regional Sales Analysis

In [ ]:
# Regional analysis
region_stats = df_clean.groupby('region').agg({
    'total_amount': ['sum', 'mean', 'count'],
    'customer_id': 'nunique'
}).round(2)
region_stats.columns = ['Total Revenue', 'Avg Order Value', 'Order Count', 'Unique Customers']
region_stats = region_stats.sort_values('Total Revenue', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Revenue by region
region_stats['Total Revenue'].sort_values(ascending=True).plot(kind='barh', ax=axes[0], color='mediumseagreen')
axes[0].set_title('Total Revenue by Region', fontweight='bold')
axes[0].set_xlabel('Revenue ($)')

# Average order value by region
region_stats['Avg Order Value'].sort_values(ascending=True).plot(kind='barh', ax=axes[1], color='gold')
axes[1].set_title('Average Order Value by Region', fontweight='bold')
axes[1].set_xlabel('Avg Order Value ($)')

plt.tight_layout()
plt.show()

print('=== REGIONAL STATISTICS ===')
display(region_stats)

### Observations - Regional Analysis

- Top performing region: [name] with $[amount] revenue
- Region with highest AOV: [name] at $[amount]
- Regional disparity: [describe differences]

## 10. Additional Visualization - Discount Impact Analysis

In [ ]:
# Discount impact on sales
discount_analysis = df_clean.groupby('discount').agg({
    'total_amount': ['sum', 'mean', 'count'],
    'quantity': 'mean'
}).round(2)
discount_analysis.columns = ['Total Revenue', 'Avg Order Value', 'Order Count', 'Avg Quantity']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

discount_analysis['Total Revenue'].plot(kind='bar', ax=axes[0], color='lightcoral')
axes[0].set_title('Total Revenue by Discount Level', fontweight='bold')
axes[0].set_xlabel('Discount')
axes[0].set_ylabel('Revenue ($)')
axes[0].tick_params(axis='x', rotation=0)

discount_analysis['Avg Order Value'].plot(kind='bar', ax=axes[1], color='lightblue')
axes[1].set_title('Average Order Value by Discount Level', fontweight='bold')
axes[1].set_xlabel('Discount')
axes[1].set_ylabel('Avg Order Value ($)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print('=== DISCOUNT IMPACT ANALYSIS ===')
display(discount_analysis)

### Observations - Discount Analysis

- Discount impact on revenue: [describe relationship]
- Optimal discount level: [identify if applicable]
- Trade-off between volume and margin: [describe]

## 11. Conclusion & Business Recommendations

### Key Findings Summary

1. **Sales Trends**: [Summarize monthly/quarterly patterns]
2. **Customer Demographics**: [Summarize age/gender insights]
3. **Product Performance**: [Top products and categories]
4. **Regional Performance**: [Regional disparities]
5. **Discount Strategy**: [Impact of discounts]
6. **Correlations**: [Key variable relationships]

### Three Actionable Business Recommendations

1. **Recommendation 1**: [Specific, data-driven action with expected impact]
   - *Based on*: [Which finding supports this]
   - *Expected outcome*: [Quantifiable if possible]

2. **Recommendation 2**: [Specific, data-driven action with expected impact]
   - *Based on*: [Which finding supports this]
   - *Expected outcome*: [Quantifiable if possible]

3. **Recommendation 3**: [Specific, data-driven action with expected impact]
   - *Based on*: [Which finding supports this]
   - *Expected outcome*: [Quantifiable if possible]